# Tarea 2 - Taller de Deep Learning

**Fecha de entrega:** 22/11/2026  
**Puntaje máximo:** 15 puntos  
**Alumno(s):**
- [Nombre1 y Apellido1]
- [Nombre2 y Apellido2]
- [Nombre3 y Apellido3]


## Introducción

El objetivo de esta tarea es evaluar su capacidad para aplicar **redes neuronales recurrentes** a un problema de clasificación de secuencias. En particular, vamos a trabajar en **reconocimiento de palabras habladas**: a partir de un clip de audio de un segundo, el modelo debe identificar qué palabra se pronunció.

### Dataset

El dataset a utilizar es **Speech Commands v0.02**, publicado por Google bajo licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Contiene 105.829 clips de audio de hasta un segundo, grabados por más de 2.600 personas con sus propios micrófonos. Cada clip contiene una única palabra en inglés, de un total de 35:

`backward`, `bed`, `bird`, `cat`, `dog`, `down`, `eight`, `five`, `follow`, `forward`, `four`, `go`, `happy`, `house`, `learn`, `left`, `marvin`, `nine`, `no`, `off`, `on`, `one`, `right`, `seven`, `sheila`, `six`, `stop`, `three`, `tree`, `two`, `up`, `visual`, `wow`, `yes`, `zero`.

El dataset se descarga desde http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz (2,3 GB comprimido, 3,3 GB descomprimido). Al descomprimirlo encontrarán:

- Una carpeta por palabra, con archivos `.wav` mono de 16 kHz y 16 bits. El nombre de cada archivo tiene la forma `<hablante>_nohash_<n>.wav`, donde `<hablante>` identifica a la persona que grabó el clip.
- Los archivos `validation_list.txt` y `testing_list.txt`, con las rutas de los clips de validación y de test.
- La carpeta `_background_noise_/`, con grabaciones largas de ruido (ambientes domésticos, ruido blanco y ruido rosa). Estas grabaciones no tienen etiqueta y no forman parte de ninguna de las 35 clases.
- Los archivos `README.md` y `LICENSE`.

La descarga y descompresión deben quedar en el notebook, de forma que quien corrija pueda reproducir todo el pipeline desde cero. El dataset se describe en el siguiente artículo, que deben citar en la entrega:

> P. Warden. *Speech Commands: A Dataset for Limited-Vocabulary Speech Recognition*. arXiv:1804.03209, 2018. https://arxiv.org/abs/1804.03209

### Partición de datos

Speech Commands define una partición oficial, que deben usar **sin modificarla**:

- Los clips listados en `validation_list.txt` forman el conjunto de **validación**.
- Los clips listados en `testing_list.txt` forman el conjunto de **test**.
- El resto de los clips forma el conjunto de **entrenamiento**.

La partición oficial asigna todos los clips de una misma persona a un único conjunto, de modo que el modelo se evalúa con voces que no escuchó durante el entrenamiento. Todas las decisiones de diseño (representación del audio, arquitectura, hiperparámetros, early stopping, selección de modelo) se toman con el conjunto de validación. El conjunto de test se usa **únicamente** para reportar las métricas finales del modelo elegido, y se evalúa una sola vez, en la sección 8. Ajustar hiperparámetros o seleccionar modelos usando el conjunto de test será penalizado.

### Tarea

Tienen libertad sobre cómo implementar y resolver el problema, así como sobre las técnicas y herramientas que quieran usar, siempre que cumplan las restricciones indicadas más abajo. Recomendamos usar Google Colab por simplicidad, pero pueden implementarlo en sus máquinas si así lo prefieren. La entrega debe realizarse en formato `.ipynb` (Jupyter Notebook) **con las celdas ya ejecutadas**.

Un clip de un segundo tiene 16.000 muestras. Recorrerlo con una red recurrente de a una muestra por paso es costoso y difícil de entrenar. Parte central de la tarea es decidir cómo presentar cada clip a la red como una secuencia (cuántos pasos tiene y qué información contiene cada paso) y justificar esa decisión con la exploración de los datos y con experimentos. Deben comparar en validación al menos dos representaciones distintas, con la misma arquitectura.

### Restricciones

- No se permite utilizar modelos pre-entrenados; cada modelo debe ser implementado desde cero.
- **El modelo final debe ser recurrente.** Las capas que combinan información de distintos instantes de tiempo deben ser recurrentes: `nn.RNN`, `nn.LSTM`, `nn.GRU` o una celda implementada por ustedes, en cualquiera de sus variantes (apiladas, bidireccionales, etc.). Se permite procesar cada paso por separado antes de la red recurrente (por ejemplo, con una capa lineal o un MLP aplicado a cada paso) y combinar las salidas de la red recurrente para clasificar. No se permiten convoluciones a lo largo del eje temporal ni Transformers.
- **Comparación de celdas recurrentes.** Deben entrenar al menos una RNN simple (`nn.RNN`), una LSTM y una GRU con la misma representación de entrada y el resto de la configuración lo más parecida posible (cantidad de capas, tamaño del estado oculto, optimizador, cantidad de épocas). A partir de esa comparación, y de los experimentos adicionales que consideren necesarios, deben elegir el tipo de red recurrente del modelo final y justificar la elección con métricas en validación, cantidad de parámetros, tiempo de entrenamiento y estabilidad del entrenamiento. Si alguna variante no logra aprender, deben explicar por qué.
- La **accuracy en test** del modelo final debe ser de **al menos 80 %**. Se espera que alcancen 90 % o más. Este requisito aplica al conjunto de test oficial.
- **Grabaciones propias.** El grupo debe grabar cada una de las 35 palabras, pronunciadas en inglés, **al menos una vez**. Lo ideal es que cada integrante grabe cada palabra al menos una vez. Las grabaciones deben tener el mismo formato que los clips del dataset (WAV mono de 16 kHz y 16 bits, de hasta un segundo, con una única palabra por archivo) y organizarse como `grabaciones/<palabra>/<integrante>_<n>.wav`. Estas grabaciones **no se usan para entrenar ni para tomar decisiones de diseño**: se usan únicamente para evaluar el modelo final, además del conjunto de test oficial. La grabación, la conversión de formato, el cambio de frecuencia de muestreo y el recorte de los archivos se hacen **fuera del notebook** y no deben incluirse en él. El notebook solo lee los archivos `.wav` ya listos de la carpeta `grabaciones/` y les aplica la misma representación que a los clips del dataset.
- Se requiere realizar un **análisis exploratorio de los datos**, que incluya la duración y el volumen de los clips, la distribución de clases en entrenamiento, validación y test, la cantidad de hablantes y de clips por hablante, y ejemplos de al menos 3 clases (gráficas de la señal y audio reproducible en el notebook, por ejemplo con `IPython.display.Audio`). Las decisiones de preprocesamiento y de entrenamiento deben tomarse en base a ese análisis, e indicar qué resultado del análisis justifica cada una.
- Deben fijar una **semilla** para que los resultados sean reproducibles, y reportar el hardware utilizado (GPU/CPU) y el tiempo de entrenamiento de cada experimento.
- Pueden usar `utils.py` y todo el material dado en clase, siempre y cuando entiendan lo que están usando. Pueden ser consultados sobre cualquier parte del código entregado.

### Reporte

Deben reportar, sobre el conjunto de test, las siguientes métricas del modelo final:

- **Accuracy**.
- **Precision, recall y F1-score** con promedio **macro**.
- **Precision, recall y F1-score por clase**.
- **Matriz de confusión**, con un análisis de al menos los 3 pares de palabras que más se confunden entre sí: cantidad de errores de cada par y una hipótesis de la causa.

Sobre las grabaciones propias deben reportar la accuracy total y por integrante, qué palabras se clasificaron mal y con qué palabra se confundieron, y una comparación con los resultados en el conjunto de test oficial, con un análisis de las diferencias.

Al final del notebook debe haber una **celda de resumen** con el tipo de red recurrente del modelo final, la accuracy y el F1-score macro en el test oficial, y la accuracy en las grabaciones propias.

También se espera poder observar la evolución del modelo (loss y accuracy en entrenamiento y validación) a medida que se entrena, mediante logs y gráficas.

### Evidencia de experimentos

Deben presentar en el notebook evidencia de los experimentos. Esto incluye:

- Un mínimo de **6 experimentos** con configuraciones distintas. La comparación de celdas recurrentes cuenta dentro de estos 6.
- Curvas de loss y accuracy en entrenamiento y validación de cada experimento. Pueden agrupar las curvas de varios experimentos en una misma gráfica.
- Una **tabla resumen** en el notebook que compare los experimentos: configuración, cantidad de parámetros, métricas en validación, tiempo de entrenamiento y hardware.
- Una justificación de cuál es el modelo final elegido y por qué.

### Lecturas sugeridas

- H. Fayek (2016). [*Speech Processing for Machine Learning: Filter banks, Mel-Frequency Cepstral Coefficients (MFCCs) and What's In-Between*](https://haythamfayek.com/2016/04/21/speech-processing-for-machine-learning.html). Introducción a la representación digital de la voz y a sus transformaciones más comunes.
- R. Pascanu, T. Mikolov e Y. Bengio (2013). *On the difficulty of training recurrent neural networks*. [arXiv:1211.5063](https://arxiv.org/abs/1211.5063).
- J. Chung, C. Gulcehre, K. Cho e Y. Bengio (2014). *Empirical Evaluation of Gated Recurrent Neural Networks on Sequence Modeling*. [arXiv:1412.3555](https://arxiv.org/abs/1412.3555).

## Formato de entrega

- La tarea se realiza en **grupos de 2 o 3 personas**.
- Se entrega este notebook con **todas las celdas ejecutadas**, junto con los archivos auxiliares que use (por ejemplo `utils.py`). **No se entregan ni los pesos de los modelos ni el dataset**. Las grabaciones propias sí se entregan, en la carpeta `grabaciones/` junto al notebook. El notebook debe poder ejecutarse de principio a fin con lo entregado (`Restart & Run All`), incluyendo la descarga del dataset.
- El notebook principal **no puede superar las 30 páginas** una vez exportado a PDF.
- Se recomienda seguir la estructura sugerida más abajo. Pueden agregar subsecciones, pero no eliminar secciones.

### Uso de herramientas de IA

Se permite el uso de herramientas de IA (ChatGPT, Copilot, Claude, etc.) como apoyo. Sin embargo:

- Los alumnos son **responsables de todo lo que entregan** y deben **entender** cada parte del código y del análisis. Pueden ser consultados sobre cualquier parte de la entrega.
- Se espera que apliquen los conocimientos vistos en clase para guiar y evaluar lo que estas herramientas producen. Un notebook que no refleje los contenidos del curso, o que contenga código que el grupo no puede explicar, puede ser penalizado.

### Rúbrica

| Ítem | Puntos |
|---|---|
| Carga del dataset y análisis exploratorio | 1,5 |
| Representación del audio como secuencia | 1,5 |
| Modelo, pipeline de entrenamiento y uso de la partición | 2 |
| Comparación de RNN simple, LSTM y GRU y elección del tipo de celda | 3 |
| Evaluación en test, métricas y análisis de errores | 2 |
| Grabaciones propias y evaluación del modelo con ellas | 2 |
| Desempeño en test: 1 punto con accuracy de 90 % o más, 0,5 puntos entre 80 % y 90 % | 1 |
| Experimentos, tabla resumen y conclusiones | 2 |
| **Total** | **15** |

Una accuracy en test menor a 80 % no cumple el requisito mínimo de la tarea.


---

## 1. Setup

_Imports, instalación de dependencias, semilla y dispositivo._

## 2. Descarga y carga del dataset

_Descarga y descompresión de `speech_commands_v0.02.tar.gz`, lectura de los clips y de las listas de validación y test._

## 3. Análisis exploratorio de datos

_Duración y volumen de los clips, distribución de clases en entrenamiento, validación y test, hablantes y clips por hablante, ejemplos de la señal y audio de al menos 3 clases. Conclusiones que guían el preprocesamiento._

## 4. Representación del audio y partición train / validación / test

_Cómo se convierte cada clip en una secuencia (cantidad de pasos y contenido de cada paso), normalización y augmentation, con su justificación. Comparación en validación de al menos dos representaciones. Construcción de los conjuntos a partir de las listas oficiales y cantidad de clips de cada conjunto._

## 5. Modelos

_Definición de los modelos recurrentes (RNN simple, LSTM, GRU y las variantes que prueben)._

## 6. Entrenamiento y experimentos

_Loop de entrenamiento, registro de métricas y demás configuraciones probadas._

## 7. Comparación de celdas recurrentes

_Resultados de RNN simple, LSTM y GRU con la misma configuración: métricas en validación, cantidad de parámetros, tiempo y curvas de entrenamiento. Elección del tipo de red recurrente y justificación._

## 8. Evaluación en test

_Métricas finales del modelo elegido: accuracy, precision/recall/F1 macro y por clase, matriz de confusión y análisis de errores._

## 9. Evaluación con grabaciones propias

_Lectura de los archivos `.wav` de `grabaciones/`, misma representación que los clips del dataset (sección 4), accuracy total y por integrante, errores y comparación con el conjunto de test oficial._

## 10. Comparación de experimentos

_Tabla resumen de los experimentos (configuración, parámetros, métricas en validación, tiempo, hardware). Justificación del modelo final._

## 11. Conclusiones

_Qué funcionó, qué no, y qué harían distinto con más tiempo o recursos. Celda de resumen final con el modelo elegido, y las métricas en test y en las grabaciones propias._